#### Day 5: Data Preprocessing

In [3]:
import pandas as pd 

X_train = pd.read_csv("../data/X_train.csv")  # Load training features
X_test = pd.read_csv("../data/X_test.csv")    # Load testing features

y_train = pd.read_csv("../data/y_train.csv").squeeze("columns")  # Load training target as a Series
y_test = pd.read_csv("../data/y_test.csv").squeeze("columns")    # Load testing target as a Series

print("Training features:", X_train.shape) 
print("Testing features:", X_test.shape)   
print("Training target:", y_train.shape)   
print("Testing target:", y_test.shape)      


#.squeeze here is used to convert a df with a sngle col if without .squeeze it will come as 94987,1

Training features: (94987, 27)
Testing features: (24221, 27)
Training target: (94987,)
Testing target: (24221,)


#### Identify categorical and numerical columns

In [4]:

categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()  # Select categorical columns

numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()  # Select numerical columns

print("Categorical columns:", categorical_cols)  
print("\nNumerical columns:", numerical_cols)  

print("\nNumber of categorical columns:", len(categorical_cols))  
print("Number of numerical columns:", len(numerical_cols)) 


Categorical columns: ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'deposit_type', 'customer_type']

Numerical columns: ['lead_time', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_guests', 'has_agent', 'has_company']

Number of categorical columns: 9
Number of numerical columns: 18


we are using xtrain instead of xtest to define our preprocessng rules

We will later apply the same fitted transformations to X_test

#### one-Hot Encoding

One-Hot Encoding converts categories into separate columns containing 0 and 1.
Why use it in our project?
We have categorical columns such as hotel, country, meal, and deposit_type.

We don't want to assign arbitrary numbers such as City Hotel = 1 and Resort Hotel = 2 because that could incorrectly suggest an order between categories.

In [6]:

from sklearn.preprocessing import OneHotEncoder  
encoder = OneHotEncoder(
    handle_unknown='ignore',  # Handle unseen categories during prediction
    min_frequency=100         # Group categories with fewer than 100 training records
)

print(encoder)  # Display the encoder configuration



OneHotEncoder(handle_unknown='ignore', min_frequency=100)


Understanding the two parameters
handle_unknown='ignore'

Suppose the model was trained without seeing a particular country. When a future booking contains that country, the encoder won't raise an error. Its unknown category will be represented by zeros for that feature's encoded columns.

min_frequency=100

We previously discovered many countries with fewer than 100 bookings. This setting allows the encoder to group infrequent categories together rather than creating a separate column for every rare category.

The frequency counts will be learned from training data only when we fit the encoder later.

#### Numerical Feature Scaling

We'll use StandardScaler.

It transforms numerical values so that each feature has approximately:
- Mean = 0
- Standard deviation = 1
These statistics are calculated from the training data.

In [7]:

from sklearn.preprocessing import StandardScaler 

scaler = StandardScaler() 

print(scaler) 

StandardScaler()


#### Create a ColumnTransformer

ColumnTransformer allows us to apply different preprocessing methods to different columns in the same dataset.

In [8]:

from sklearn.compose import ColumnTransformer 

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', encoder, categorical_cols),  # Apply OneHotEncoder to categorical columns
        ('num', scaler, numerical_cols)       # Apply StandardScaler to numerical columns
    ]
)

print(preprocessor)  


ColumnTransformer(transformers=[('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               min_frequency=100),
                                 ['hotel', 'arrival_date_month', 'meal',
                                  'country', 'market_segment',
                                  'distribution_channel', 'reserved_room_type',
                                  'deposit_type', 'customer_type']),
                                ('num', StandardScaler(),
                                 ['lead_time', 'arrival_date_year',
                                  'arrival_date_week_number',
                                  'arrival_date_day_of_month',
                                  'stays_in_weekend_nights',
                                  'stays_in_week_nights', 'adults', 'children',
                                  'babies', 'is_repeated_guest',
                                  'previous_cancellations',
                   

#### Fit and transform training data

fit() learns information from the training data, such as category frequencies, numerical means, and standard deviations.



transform() uses that information to convert the data.



fit_transform() performs both operations together.

In [9]:
X_train_processed = preprocessor.fit_transform(X_train)  # Learn preprocessing rules from training data and transform it

print("Original training shape:", X_train.shape)  
print("Processed training shape:", X_train_processed.shape) 


Original training shape: (94987, 27)
Processed training shape: (94987, 103)


Why did 27 features become 103?

Because One-Hot Encoding converted our 9 categorical features into 85 encoded columns, while our 18 numerical features remained as 18 columns.

So, 85 + 18 = 103 features.

#### Transform the testing data

In [10]:

X_test_processed = preprocessor.transform(X_test)  # Apply training-based preprocessing to test data

print("Original testing shape:", X_test.shape)  # Display original test dimensions
print("Processed testing shape:", X_test_processed.shape)  # Display transformed test dimensions

print("Same number of features:", X_train_processed.shape[1] == X_test_processed.shape[1])  # Check feature counts match


Original testing shape: (24221, 27)
Processed testing shape: (24221, 103)
Same number of features: True


here tranform is used nstead of fit_transform() on testing data bcs the test datamust use the same encoding categ, mean and stndrd deviation

if we fit the preprocessing again on the test data, we would introduce data leakage and potentially produce inconsistent features.


#### check for missing values after preprocessing

In [13]:
import numpy as np  

In [14]:
print("Training missing values:", np.isnan(X_train_processed.data).sum())
print("Testing missing values:", np.isnan(X_test_processed.data).sum())

#is nan means to check if it has null values

Training missing values: 0
Testing missing values: 0


#### Save our preprocessing pipeline
Now we need to save the fitted ColumnTransformer so that we can reuse it during model training and deployment.

We'll use Joblib, a Python library commonly used to save machine learning objects.

In [16]:
import joblib

joblib.dump(preprocessor, "../models/preprocessor.pkl")  

print("Preprocessor saved successfully!") 


Preprocessor saved successfully!



## Day 5: Data Preprocessing Summary

- Loaded training and testing datasets from Day 4.
- Identified 9 categorical and 18 numerical features.
- Applied OneHotEncoder to categorical features.
- Applied StandardScaler to numerical features.
- Used ColumnTransformer to combine both transformations.
- Fitted preprocessing on training data only to prevent data leakage.
- Transformed training data: 94,987 rows × 103 features.
- Transformed testing data: 24,221 rows × 103 features.
- Verified that both processed datasets contain no missing values.
- Saved the fitted preprocessor as `models/preprocessor.pkl`.

**Result:** Data preprocessing completed successfully. The datasets are ready for machine learning model training.
